# 06 - 完整 GPT 模型 (AI Infra 视角)

本节从 **工程实现** 角度理解完整的 GPT：
- 参数量计算公式
- 显存分析 (训练 vs 推理)
- 权重初始化策略
- 模型配置与扩展规律

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

## 1. GPT 结构 (30秒版)

```
token_ids [1, 234, 567]
        ↓
   Embedding (vocab → dim)
        ↓
      Norm
        ↓
   ┌─────────┐
   │ Block 0 │ ×N 层
   │ Block 1 │
   │   ...   │
   └─────────┘
        ↓
      Norm
        ↓
    LM Head (dim → vocab)
        ↓
    logits
```

## 2. 参数量计算 (重要!)

### 公式

```
总参数 = Embedding + N × Block + LM_Head

Embedding: vocab × dim
Block:     12 × dim² (Attn 4dim² + MLP 8dim²)
LM_Head:   dim × vocab (可与 Embedding 共享)

近似公式 (忽略 Embedding):
  P ≈ 12 × N × dim²
```

In [ ]:
def count_params(n_layers, dim, vocab_size, n_kv_heads=None, n_heads=None, tie_weights=False):
    """精确计算 GPT 参数量"""
    n_heads = n_heads or dim // 64
    n_kv_heads = n_kv_heads or n_heads
    head_dim = dim // n_heads
    
    # Embedding
    embedding = vocab_size * dim
    
    # 每层 Attention (考虑 GQA)
    q_proj = dim * dim
    k_proj = dim * (n_kv_heads * head_dim)
    v_proj = dim * (n_kv_heads * head_dim)
    o_proj = dim * dim
    attn_per_layer = q_proj + k_proj + v_proj + o_proj
    
    # 每层 MLP
    mlp_per_layer = dim * 4 * dim + 4 * dim * dim
    
    # 所有 Block
    all_blocks = n_layers * (attn_per_layer + mlp_per_layer)
    
    # LM Head
    lm_head = 0 if tie_weights else dim * vocab_size
    
    total = embedding + all_blocks + lm_head
    
    return {
        'embedding': embedding,
        'attn_per_layer': attn_per_layer,
        'mlp_per_layer': mlp_per_layer,
        'all_blocks': all_blocks,
        'lm_head': lm_head,
        'total': total
    }

# LLaMA 7B 配置
params = count_params(n_layers=32, dim=4096, vocab_size=32000, n_kv_heads=32)
print("LLaMA 7B 参数分布:")
for k, v in params.items():
    print(f"  {k:18s}: {v/1e9:.2f}B ({v/params['total']*100:.1f}%)")

In [ ]:
# 不同规模模型对比
models = {
    "GPT-2 (124M)":   (12, 768, 50257),
    "GPT-2 (350M)":   (24, 1024, 50257),
    "GPT-2 (774M)":   (36, 1280, 50257),
    "GPT-2 (1.5B)":   (48, 1600, 50257),
    "LLaMA 7B":       (32, 4096, 32000),
    "LLaMA 13B":      (40, 5120, 32000),
    "LLaMA 70B":      (80, 8192, 32000),
}

print("模型规模对比:")
print(f"{'Model':<16} {'Layers':>8} {'Dim':>8} {'Params':>12}")
print("-" * 48)
for name, (n_layers, dim, vocab) in models.items():
    p = count_params(n_layers, dim, vocab)['total']
    print(f"{name:<16} {n_layers:>8} {dim:>8} {p/1e9:>10.2f}B")

## 3. 显存分析 (重要!)

### 训练显存 = 模型参数 + 梯度 + 优化器状态 + 激活值

```
混合精度训练 (BF16 + FP32 master weights):

模型参数:     2Φ bytes (BF16)
梯度:         2Φ bytes (BF16)
优化器状态:   12Φ bytes (FP32 master + momentum + variance)
激活值:       ~B × T × dim × layers × 10 bytes

总计 ≈ 16Φ + 激活值
```

In [ ]:
def training_memory(n_params, batch_size, seq_len, dim, n_layers):
    """估算训练显存 (GB)"""
    # 模型 + 梯度 + 优化器 (混合精度)
    model_grad_opt = n_params * 16  # 16 bytes per param
    
    # 激活值 (近似)
    activation = batch_size * seq_len * dim * n_layers * 10  # 约 10 bytes/element
    
    total = model_grad_opt + activation
    return {
        'model_grad_opt': model_grad_opt / 1e9,
        'activation': activation / 1e9,
        'total': total / 1e9
    }

# LLaMA 7B 训练
n_params = 7e9
mem = training_memory(n_params, batch_size=4, seq_len=4096, dim=4096, n_layers=32)

print("LLaMA 7B 训练显存 (batch=4, seq=4096):")
print(f"  模型+梯度+优化器: {mem['model_grad_opt']:.1f} GB")
print(f"  激活值:           {mem['activation']:.1f} GB")
print(f"  总计:             {mem['total']:.1f} GB")
print(f"\n需要 {mem['total']/80:.1f} 张 H100 (80GB)")

In [ ]:
def inference_memory(n_params, batch_size, seq_len, n_kv_heads, head_dim, n_layers):
    """估算推理显存 (GB)"""
    # 模型参数 (BF16)
    model = n_params * 2
    
    # KV Cache
    kv_cache = 2 * batch_size * seq_len * n_kv_heads * head_dim * n_layers * 2
    
    return {
        'model': model / 1e9,
        'kv_cache': kv_cache / 1e9,
        'total': (model + kv_cache) / 1e9
    }

# LLaMA 7B 推理
mem = inference_memory(7e9, batch_size=32, seq_len=4096, n_kv_heads=32, head_dim=128, n_layers=32)

print("LLaMA 7B 推理显存 (batch=32, seq=4096):")
print(f"  模型参数:  {mem['model']:.1f} GB")
print(f"  KV Cache:  {mem['kv_cache']:.1f} GB")
print(f"  总计:      {mem['total']:.1f} GB")

## 4. 权重初始化

### nanochat 的初始化策略

```python
# Linear 层
std = (1/√fan_in) × min(1, √(fan_out/fan_in))

# 残差分支的输出投影: 初始化为 0
# 这样初始时 f(x)=0，残差路径是恒等映射
```

In [ ]:
import math

def init_weights(module):
    """nanochat 的权重初始化"""
    if isinstance(module, nn.Linear):
        fan_in = module.weight.size(1)
        fan_out = module.weight.size(0)
        std = 1.0 / math.sqrt(fan_in) * min(1.0, math.sqrt(fan_out / fan_in))
        nn.init.normal_(module.weight, mean=0.0, std=std)
    elif isinstance(module, nn.Embedding):
        nn.init.normal_(module.weight, mean=0.0, std=1.0)

# 关键: 残差分支输出初始化为 0
def zero_init_residual(model):
    """把残差分支的输出投影初始化为 0"""
    for block in model.blocks:
        nn.init.zeros_(block.attn.c_proj.weight)
        nn.init.zeros_(block.mlp.c_proj.weight)

print("初始化为 0 的好处:")
print("  初始时 f(x)=0，y = x + f(x) = x")
print("  模型开始时是恒等映射")
print("  训练更稳定，尤其是深层网络")

## 5. 模型配置

In [ ]:
@dataclass
class GPTConfig:
    vocab_size: int = 50304    # 词表大小 (64 的倍数，利于 GPU)
    sequence_len: int = 1024   # 最大序列长度
    n_layer: int = 12          # Block 层数
    n_head: int = 12           # 注意力头数
    n_kv_head: int = 12        # KV 头数 (GQA)
    n_embd: int = 768          # 嵌入维度

# Scaling Laws: 参数量、数据量、计算量的关系
print("Scaling Laws 经验规律:")
print("  - 参数翻倍，性能提升约 log(2) ≈ 0.3")
print("  - 计算最优: tokens ≈ 20 × params")
print("  - LLaMA: 7B 模型用 1T tokens 训练")

## 6. 损失函数

```
Cross Entropy Loss:
  L = -log(P(correct_token))
  
随机初始化时:
  L ≈ log(vocab_size)
  vocab=50000 → L ≈ 10.8

训练目标:
  GPT-3: L ≈ 3.0
  GPT-4: L ≈ 2.5 (估计)
```

In [ ]:
# 损失计算
def compute_loss(logits, targets):
    """
    logits: (B, T, vocab_size)
    targets: (B, T)
    """
    B, T, V = logits.shape
    return F.cross_entropy(
        logits.view(B * T, V),
        targets.view(B * T)
    )

# 随机初始化的损失
vocab_size = 50000
expected_loss = torch.log(torch.tensor(vocab_size)).item()
print(f"随机初始化的期望损失: {expected_loss:.2f}")
print(f"训练后目标损失: ~3.0 (perplexity ~20)")

## 7. 面试常见问题

### Q1: 如何估算模型参数量?

**答**:
- 近似公式: P ≈ 12 × N × dim²
- N=32, dim=4096: P ≈ 12 × 32 × 4096² ≈ 6.4B
- 加上 Embedding (~0.5B) ≈ 7B

---

### Q2: 训练 7B 模型需要多少显存?

**答**:
- 模型+梯度+优化器: 16 × 7B = 112GB
- 激活值: 约 20-50GB (取决于 batch)
- 总计: 130-160GB
- 需要 2-4 张 H100 (80GB)

---

### Q3: 推理时主要显存开销是什么?

**答**:
- 模型参数: 2Φ bytes (BF16)
- KV Cache: 随序列长度增长
- 长序列时 KV Cache 可能超过模型本身

---

### Q4: 为什么 vocab_size 通常是 64 的倍数?

**答**:
- GPU Tensor Core 对 64 对齐的矩阵效率最高
- 50257 (GPT-2) → 50304 (nanochat)
- padding 几个 token 就能获得 ~5% 加速

---

### Q5: 权重初始化为什么重要?

**答**:
- 初始化不当 → 梯度消失/爆炸
- 残差分支初始化为 0 → 初始时恒等映射
- std 与 fan_in 相关 → 保持激活值方差稳定

---

### Q6: 什么是 Scaling Laws?

**答**:
- 模型性能与参数量、数据量、计算量的关系
- L ∝ 1/P^0.076 (参数)
- L ∝ 1/D^0.095 (数据)
- 计算最优: tokens ≈ 20 × params

## 8. 总结速查表

| 主题 | 要点 |
|------|------|
| **参数公式** | P ≈ 12 × N × dim² |
| **训练显存** | 16Φ + 激活值 (混合精度) |
| **推理显存** | 2Φ + KV Cache |
| **初始化** | 残差分支输出 = 0 |
| **vocab 对齐** | 64 的倍数，利于 GPU |
| **Scaling** | tokens ≈ 20 × params |

### 显存速算

```
7B 模型训练: ~150GB (需要 2×H100)
7B 模型推理: ~14GB + KV Cache
70B 模型训练: ~1.5TB (需要 20×H100)
70B 模型推理: ~140GB (需要 2×H100)
```